# 📊 Day 3 – Classical Portfolio Theory: Mean-Variance Optimization
## Quantum Portfolio Optimization: Efficient Frontier & Binary vs Continuous Gap

### Learning Objectives for Day 3:
1. **Implement Mean-Variance optimization** (Markowitz model) using continuous weights
2. **Generate the efficient frontier** — set of optimal risk-return combinations
3. **Understand the binary selection gap** — why QUBO uses 0/1 vs continuous 0-1 weights

### Key Concepts:
- **Continuous Allocation**: Weights $w_i \in [0,1]$ with $\sum w_i = 1$ (classical finance)
- **Binary Selection**: $x_i \in \{0,1\}$ with $\sum x_i = k$ (QUBO for quantum)
- **Efficient Frontier**: Curve of optimal portfolios (max return for given risk, or min risk for given return)

### Why This Matters for QAOA:
Classical methods provide the **ground truth** baseline. The QUBO formulation approximates this with binary constraints, trading optimality for quantum solvability.

---


## Section 1: Import Required Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import os

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("✔ All libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

✔ All libraries imported successfully!
NumPy version: 2.0.2
Pandas version: 2.2.2


## Section 2: Load Portfolio Data from Day 2

Load the annualized mean returns (μ) and covariance matrix (Σ) computed in Day 2.

In [9]:
# Load data from Day 2 cached results (hardcoded for notebook compatibility)
# In a local environment, this would load from '../data/cached/'

# Mean returns (μ) - annualized expected returns
mu = np.array([ 0.06574643,  0.0016126,   0.05589102, -0.00123439,  0.16337352, -0.35367911])

# Covariance matrix (Σ) - annualized covariance
sigma = np.array([
    [0.03706484, 0.01711054, 0.02571261, 0.02044438, 0.02369213, 0.01307968],
    [0.01711054, 0.0565081,  0.02112788, 0.01885764, 0.02126277, 0.01829955],
    [0.02571261, 0.02112788, 0.05814683, 0.02445718, 0.03337956, 0.0280938 ],
    [0.02044438, 0.01885764, 0.02445718, 0.03741472, 0.02076008, 0.01852383],
    [0.02369213, 0.02126277, 0.03337956, 0.02076008, 0.0646312,  0.03324001],
    [0.01307968, 0.01829955, 0.0280938,  0.01852383, 0.03324001, 0.15364106]
])

# Stock names
stock_names = ['ICICIBANK', 'KOTAKBANK', 'AXISBANK', 'HDFCBANK', 'SBIN', 'INDUSINDBK']

print("=" * 70)
print("PORTFOLIO DATA LOADED FROM DAY 2")
print("=" * 70)
print(f"✔ Number of assets: {len(stock_names)}")
print(f"✔ Assets: {', '.join(stock_names)}")
print(f"✔ μ shape: {mu.shape} (expected returns)")
print(f"✔ Σ shape: {sigma.shape} (covariance matrix)")

print("\nAnnualized Expected Returns (μ):")
for name, ret in zip(stock_names, mu):
    print(f"  {name:12}: {ret:>8.4f}")

print(f"\nAnnualized Covariance Matrix (Σ) preview:")
print(pd.DataFrame(sigma, index=stock_names, columns=stock_names).round(4))
print("=" * 70)

PORTFOLIO DATA LOADED FROM DAY 2
✔ Number of assets: 6
✔ Assets: ICICIBANK, KOTAKBANK, AXISBANK, HDFCBANK, SBIN, INDUSINDBK
✔ μ shape: (6,) (expected returns)
✔ Σ shape: (6, 6) (covariance matrix)

Annualized Expected Returns (μ):
  ICICIBANK   :   0.0657
  KOTAKBANK   :   0.0016
  AXISBANK    :   0.0559
  HDFCBANK    :  -0.0012
  SBIN        :   0.1634
  INDUSINDBK  :  -0.3537

Annualized Covariance Matrix (Σ) preview:
            ICICIBANK  KOTAKBANK  AXISBANK  HDFCBANK    SBIN  INDUSINDBK
ICICIBANK      0.0371     0.0171    0.0257    0.0204  0.0237      0.0131
KOTAKBANK      0.0171     0.0565    0.0211    0.0189  0.0213      0.0183
AXISBANK       0.0257     0.0211    0.0581    0.0245  0.0334      0.0281
HDFCBANK       0.0204     0.0189    0.0245    0.0374  0.0208      0.0185
SBIN           0.0237     0.0213    0.0334    0.0208  0.0646      0.0332
INDUSINDBK     0.0131     0.0183    0.0281    0.0185  0.0332      0.1536


## Section 3: Mean-Variance Optimization (Markowitz Model)

### The Optimization Problem:
**Minimize portfolio variance** subject to:
- Target return: $w^T \mu = \mu_{target}$
- Weights sum to 1: $w^T 1 = 1$
- No short selling: $w_i \geq 0$ for all i

**Mathematical formulation:**
$$\min_w \frac{1}{2} w^T \Sigma w$$
$$\text{subject to: } w^T \mu = \mu_{target}, \quad w^T 1 = 1, \quad w \geq 0$$

We'll use scipy.optimize.minimize with Sequential Least Squares Programming (SLSQP).

In [ ]:
def portfolio_variance(w, sigma):
    """Calculate portfolio variance: w^T Σ w"""
    return w.T @ sigma @ w

def portfolio_return(w, mu):
    """Calculate portfolio expected return: w^T μ"""
    return w.T @ mu

def optimize_portfolio(mu, sigma, target_return):
    """
    Solve the Markowitz mean-variance optimization problem.
    Minimize variance subject to target return and full investment.
    """
    n_assets = len(mu)

    # Initial guess: equal weights
    w0 = np.ones(n_assets) / n_assets

    # Constraints
    constraints = [
        {'type': 'eq', 'fun': lambda w: w.sum() - 1},  # weights sum to 1
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, mu) - target_return}  # target return
    ]

    # Bounds: no short selling (w_i >= 0)
    bounds = [(0, 1) for _ in range(n_assets)]

    # Minimize portfolio variance
    result = minimize(
        portfolio_variance,
        w0,
        args=(sigma,),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'disp': False}
    )

    if result.success:
        w_opt = result.x
        variance_opt = portfolio_variance(w_opt, sigma)
        return_opt = portfolio_return(w_opt, mu)
        return w_opt, return_opt, variance_opt
    else:
        print(f"Optimization failed for target return {target_return:.4f}")
        return None, None, None

# Test the optimization with a sample target return
target_return = 0.05  # 5% annual return
w_opt, ret_opt, var_opt = optimize_portfolio(mu, sigma, target_return)

if w_opt is not None:
    print("=" * 70)
    print("MEAN-VARIANCE OPTIMIZATION TEST")
    print("=" * 70)
    print(f"Target return: {target_return:.4f}")
    print(f"Optimized return: {ret_opt:.4f}")
    print(f"Optimized variance: {var_opt:.6f}")
    print(f"Optimized volatility (std dev): {np.sqrt(var_opt):.4f}")

    print("\nOptimal weights:")
    for name, weight in zip(stock_names, w_opt):
        print(f"  {name:12}: {weight:>8.4f}")
    print("=" * 70)
else:
    print("Optimization failed!")

## Section 4: Generate the Efficient Frontier

The **efficient frontier** is the set of optimal portfolios that offer the highest expected return for a given level of risk (or lowest risk for a given return).

### Algorithm:
1. Define a range of target returns (from min to max possible)
2. For each target return, find the minimum variance portfolio
3. Plot the (volatility, return) pairs

This gives us the **Markowitz bullet** — the boundary of optimal portfolios.

In [ ]:
def generate_efficient_frontier(mu, sigma, n_points=50):
    """
    Generate points on the efficient frontier by solving for minimum variance
    portfolios across a range of target returns.
    """
    # Define range of target returns
    min_return = np.min(mu)
    max_return = np.max(mu)
    target_returns = np.linspace(min_return, max_return, n_points)

    # Store results
    portfolios = []

    for target_ret in target_returns:
        w_opt, ret_opt, var_opt = optimize_portfolio(mu, sigma, target_ret)
        if w_opt is not None:
            portfolios.append({
                'return': ret_opt,
                'variance': var_opt,
                'volatility': np.sqrt(var_opt),
                'weights': w_opt
            })

    return portfolios

# Generate efficient frontier
print("Generating efficient frontier...")
efficient_frontier = generate_efficient_frontier(mu, sigma, n_points=30)

if efficient_frontier:
    print(f"✔ Generated {len(efficient_frontier)} portfolios on the efficient frontier")

    # Extract data for plotting
    returns = [p['return'] for p in efficient_frontier]
    volatilities = [p['volatility'] for p in efficient_frontier]

    # Plot the efficient frontier
    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot efficient frontier
    ax.plot(volatilities, returns, 'b-', linewidth=3, label='Efficient Frontier')

    # Plot individual assets
    asset_volatilities = np.sqrt(np.diag(sigma))
    ax.scatter(asset_volatilities, mu, c='red', s=100, alpha=0.7, label='Individual Assets')

    # Add asset labels
    for i, name in enumerate(stock_names):
        ax.annotate(name, (asset_volatilities[i], mu[i]),
                   xytext=(5, 5), textcoords='offset points', fontsize=10)

    ax.set_xlabel('Annualized Volatility (Risk)', fontsize=12)
    ax.set_ylabel('Annualized Expected Return', fontsize=12)
    ax.set_title('Day 3 – Efficient Frontier\nClassical Mean-Variance Optimization (Bank Nifty Stocks)',
                fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nEfficient Frontier Summary:")
    print("-" * 40)
    print(f"Minimum return: {min(returns):.4f}")
    print(f"Maximum return: {max(returns):.4f}")
    print(f"Minimum volatility: {min(volatilities):.4f}")
    print(f"Maximum volatility: {max(volatilities):.4f}")
else:
    print("Failed to generate efficient frontier!")

## Section 5: The Binary vs Continuous Allocation Gap

### ⚠️ Critical Gap: Binary Selection vs Continuous Weights

**Classical Finance (Continuous):**
- **Weights**: $w_i \in [0, 1]$ with $\sum w_i = 1$
- **Optimization**: Quadratic programming (QP) solvable in polynomial time
- **Result**: Any point on the efficient frontier (optimal risk-return tradeoffs)
- **Flexibility**: Fine-grained control over portfolio composition

**Quantum Portfolio Optimization (Binary):**
- **Selection**: $x_i \in \{0, 1\}$ with $\sum x_i = k$ (fixed number of assets)
- **Optimization**: QUBO (Quadratic Unconstrained Binary Optimization)
- **Result**: Only discrete points; cannot achieve continuous frontier
- **Constraint**: Must select exactly k assets (e.g., k=3 out of 6)

### Why This Gap Matters:
1. **Optimality Loss**: Binary constraints prevent reaching the true efficient frontier
2. **Practical Impact**: Real portfolios use continuous weights; binary is an approximation
3. **Quantum Advantage**: QUBO enables quantum speedup, but at the cost of discretization

### Real-World Relevance:
- **Institutional portfolios**: Use continuous weights (e.g., 15.7% in AAPL)
- **Retail robo-advisors**: Often use discrete allocations (e.g., 20% stocks, 80% bonds)
- **Our QAOA approach**: Binary selection for quantum feasibility

### Next Steps (Day 4):
- Formulate the QUBO problem using μ and Σ
- Implement brute-force solver for ground truth
- Compare quantum QAOA results against this classical baseline

---
**Key Takeaway**: Classical methods give the theoretical optimum. Quantum methods provide practical approximations with potential speedup for large portfolios.

## Section 6: QUBO Formulation & Brute Force Baseline (Day 11)

Now we switch from continuous weights to **binary selection** — the exact same formulation that QAOA solves on a quantum computer.

We define `build_Q_matrix` right here so this notebook is **100% self-contained** and runs on Google Colab without any external imports.


In [ ]:
import itertools
import time

# ─── QUBO Builder (copied from qubo/qubo_builder.py) ───
def build_Q_matrix(returns, cov, penalty, k, risk_weight=1.0):
    """Build the QUBO Q matrix for the portfolio problem."""
    n = len(returns)
    Q = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                Q[i][i] = -returns[i] + risk_weight * cov[i][i] + penalty * (1 - 2 * k)
            else:
                Q[i][j] = risk_weight * cov[i][j] + penalty
    Q = (Q + Q.T) / 2
    return Q

# ─── Build Q matrix ───
k = 2          # pick exactly 2 stocks
penalty = 10.0
Q = build_Q_matrix(mu, sigma, penalty, k)
print("QUBO Matrix Q built!")
print(f"Shape: {Q.shape}")

# ─── Brute Force (Ground Truth) ───
start_time = time.time()
n = len(mu)
best_obj_bf = float('inf')
best_x_bf = None

for bits in itertools.product([0, 1], repeat=n):
    x = np.array(bits)
    obj = float(x @ Q @ x)
    if obj < best_obj_bf:
        best_obj_bf = obj
        best_x_bf = x

bf_time = time.time() - start_time
selected_bf = [stock_names[i] for i in range(n) if best_x_bf[i] == 1]
print(f"\nBrute Force Best Bitstring: {best_x_bf}")
print(f"Brute Force Best Portfolio: {selected_bf}")
print(f"Brute Force QUBO Value:     {best_obj_bf:.4f}")
print(f"Time taken:                 {bf_time:.6f} seconds")


## Section 7: Greedy Selection

The greedy algorithm iteratively adds the single stock that **reduces the QUBO energy the most**, until exactly `k` stocks are selected. It is fast but myopic — it cannot undo past choices.


In [ ]:
# ─── Greedy QUBO Search (copied from classical/greedy.py) ───
def greedy_qubo_search(Q, k):
    """Greedy heuristic: iteratively pick the asset that minimises objective."""
    start_time = time.time()
    n = len(Q)
    x = np.zeros(n, dtype=int)
    for _ in range(k):
        best_gain = float("inf")
        best_idx = -1
        for i in range(n):
            if x[i] == 0:
                x_test = x.copy()
                x_test[i] = 1
                obj = float(x_test @ Q @ x_test)
                if obj < best_gain:
                    best_gain = obj
                    best_idx = i
        if best_idx != -1:
            x[best_idx] = 1
    return x, float(x @ Q @ x), time.time() - start_time

# ─── Run Greedy ───
print("Running Greedy QUBO Search...")
x_greedy, obj_greedy, greedy_time = greedy_qubo_search(Q, k)
selected_greedy = [stock_names[i] for i in range(len(stock_names)) if x_greedy[i] == 1]
print(f"Greedy Best Bitstring:  {x_greedy}")
print(f"Greedy Best Portfolio:  {selected_greedy}")
print(f"Greedy QUBO Value:      {obj_greedy:.4f}")
print(f"Time taken:             {greedy_time:.6f} seconds")


## Section 8: Simulated Annealing

Simulated Annealing escapes local minima by **occasionally accepting worse solutions** early on (high temperature), then gradually becoming strict (cooling). It swaps one stock in for one stock out to maintain exactly `k` selections.


In [ ]:
import math

# ─── Simulated Annealing (copied from classical/sim_annealing.py) ───
def simulated_annealing_qubo(Q, k, T_start=1000.0, cooling_rate=0.99, max_iter=10000):
    """SA heuristic for QUBO. Maintains exactly k selected assets."""
    start_time = time.time()
    n = len(Q)
    x_current = np.zeros(n, dtype=int)
    random_indices = np.random.choice(n, k, replace=False)
    x_current[random_indices] = 1
    current_obj = float(x_current @ Q @ x_current)

    best_x = x_current.copy()
    best_obj = current_obj
    T = T_start

    for _ in range(max_iter):
        if T < 1e-8:
            break
        x_neighbor = x_current.copy()
        ones_idx = np.where(x_neighbor == 1)[0]
        zeros_idx = np.where(x_neighbor == 0)[0]
        if len(ones_idx) > 0 and len(zeros_idx) > 0:
            x_neighbor[np.random.choice(ones_idx)] = 0
            x_neighbor[np.random.choice(zeros_idx)] = 1

        neighbor_obj = float(x_neighbor @ Q @ x_neighbor)
        delta = neighbor_obj - current_obj
        if delta < 0 or np.random.rand() < math.exp(-delta / T):
            x_current = x_neighbor.copy()
            current_obj = neighbor_obj
            if current_obj < best_obj:
                best_obj = current_obj
                best_x = x_current.copy()
        T *= cooling_rate

    return best_x, best_obj, time.time() - start_time

# ─── Run Simulated Annealing ───
print("Running Simulated Annealing...")
np.random.seed(42)
x_sa, obj_sa, sa_time = simulated_annealing_qubo(Q, k, T_start=1000.0, cooling_rate=0.99, max_iter=10000)
selected_sa = [stock_names[i] for i in range(len(stock_names)) if x_sa[i] == 1]
print(f"SA Best Bitstring:  {x_sa}")
print(f"SA Best Portfolio:  {selected_sa}")
print(f"SA QUBO Value:      {obj_sa:.4f}")
print(f"Time taken:         {sa_time:.6f} seconds")


## Section 9: QAOA (Quantum Approximate Optimization)

Now we run the **exact same QUBO** through our quantum algorithm on a local simulator. The QAOA circuit, Ising conversion, and optimizer are all defined below — no external files needed.


In [ ]:
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz

# ─── QUBO → Ising Hamiltonian (copied from qaoa/qaoa_circuit.py) ───
def qubo_to_ising(Q):
    n = len(Q)
    pauli_list = []
    offset = 0.0
    for i in range(n):
        offset += Q[i][i] / 2.0
        ps = ['I'] * n
        ps[n - 1 - i] = 'Z'
        pauli_list.append((''.join(ps), -Q[i][i] / 2.0))
    for i in range(n):
        for j in range(i + 1, n):
            c = Q[i][j] + Q[j][i]
            offset += c / 4.0
            pi, pj, pij = ['I']*n, ['I']*n, ['I']*n
            pi[n-1-i], pj[n-1-j] = 'Z', 'Z'
            pij[n-1-i], pij[n-1-j] = 'Z', 'Z'
            pauli_list.extend([(''.join(pi), -c/4), (''.join(pj), -c/4), (''.join(pij), c/4)])
    h = SparsePauliOp.from_list(pauli_list).simplify()
    return SparsePauliOp(h.paulis, coeffs=h.coeffs.real), offset

# ─── Run QAOA ───
print("Building QAOA Circuit (p=4)...")
qaoa_start = time.time()
p = 4
hamiltonian, offset = qubo_to_ising(Q)
qaoa_circuit = QAOAAnsatz(cost_operator=hamiltonian, reps=p)

estimator = StatevectorEstimator()

def evaluate_expectation(params):
    result = estimator.run([(qaoa_circuit, hamiltonian, params)]).result()[0]
    return result.data.evs

np.random.seed(42)
initial_point = np.random.rand(qaoa_circuit.num_parameters) * np.pi
res = minimize(evaluate_expectation, initial_point, method='COBYLA')

optimized_circuit = qaoa_circuit.assign_parameters(res.x)
optimized_circuit.measure_all()

sampler = StatevectorSampler()
raw_counts = sampler.run([optimized_circuit]).result()[0].data.meas.get_counts()
counts = {bs[::-1]: v for bs, v in raw_counts.items()}  # fix endianness
best_bs = max(counts, key=counts.get)
best_x_qaoa = np.array(list(best_bs), dtype=int)
obj_qaoa = float(best_x_qaoa @ Q @ best_x_qaoa)
qaoa_time = time.time() - qaoa_start

selected_qaoa = [stock_names[i] for i in range(len(stock_names)) if best_x_qaoa[i] == 1]
print(f"QAOA Best Bitstring:  {best_x_qaoa}")
print(f"QAOA Best Portfolio:  {selected_qaoa}")
print(f"QAOA QUBO Value:      {obj_qaoa:.4f}")
print(f"Time taken:           {qaoa_time:.4f} seconds")


## Section 10: Performance Comparison Table (Day 12)

The ultimate benchmark — all four methods evaluated on the **exact same QUBO matrix**, measuring Return, Risk, QUBO Value, Execution Time, and Distance from the Brute Force optimum.


In [ ]:
# ─── Compute return & risk for each method ───
def get_portfolio_metrics(x):
    ret  = float(x @ mu)          # expected return
    risk = float(x @ sigma @ x)   # variance (risk)
    return ret, risk

bf_ret,  bf_risk  = get_portfolio_metrics(best_x_bf)
gr_ret,  gr_risk  = get_portfolio_metrics(x_greedy)
sa_ret,  sa_risk  = get_portfolio_metrics(x_sa)
qa_ret,  qa_risk  = get_portfolio_metrics(best_x_qaoa)

def dist_pct(val):
    return abs((val - best_obj_bf) / best_obj_bf) * 100 if best_obj_bf != 0 else 0

results = pd.DataFrame([
    {"Method": "Brute Force (Ground Truth)", "Return": bf_ret, "Risk (Var)": bf_risk, "QUBO Value": best_obj_bf, "Time (s)": bf_time, "Dist from Optimal (%)": 0.0},
    {"Method": "Greedy Selection",           "Return": gr_ret, "Risk (Var)": gr_risk, "QUBO Value": obj_greedy,  "Time (s)": greedy_time, "Dist from Optimal (%)": dist_pct(obj_greedy)},
    {"Method": "Simulated Annealing",        "Return": sa_ret, "Risk (Var)": sa_risk, "QUBO Value": obj_sa,      "Time (s)": sa_time, "Dist from Optimal (%)": dist_pct(obj_sa)},
    {"Method": "QAOA (Simulator, p=4)",      "Return": qa_ret, "Risk (Var)": qa_risk, "QUBO Value": obj_qaoa,    "Time (s)": qaoa_time, "Dist from Optimal (%)": dist_pct(obj_qaoa)},
])

display(results)

# Save to results directory
import os
os.makedirs('../results', exist_ok=True)
results.to_csv('../results/metrics.csv', index=False)
print("\nSaved to ../results/metrics.csv")
